# Final Test Evaluation — Template

Run only after architecture, hyperparameters, threshold, and checkpoint are frozen using validation data. Adapt preprocessing to match the selected training notebook.


In [ ]:
from pathlib import Path
import json, cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

DATASET_ROOT = Path("/raid/home/dgx1725/Warwick_QU_Dataset_Binary_FIXED")
MODEL_PATH = Path("/raid/home/dgx1725/REPLACE_WITH_SELECTED_MODEL.keras")
OUTPUT_DIR = Path("/raid/home/dgx1725/final_test_evaluation/REPLACE_WITH_MODEL_NAME")
IMG_SIZE = (256, 384)
PRED_THRESHOLD = 0.5
BOUNDARY_TOLERANCE_PX = 2
VALID_EXTS = {'.png', '.bmp', '.bm', '.jpg', '.jpeg', '.tif', '.tiff'}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'predicted_masks').mkdir(exist_ok=True)
assert MODEL_PATH.exists(), f'Checkpoint not found: {MODEL_PATH}'


## 1. Validate test pairs


In [ ]:
def files_in(folder):
    folder = Path(folder)
    if not folder.is_dir(): raise FileNotFoundError(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS)

def canonical_mask_stem(path):
    stem = Path(path).stem
    return stem[:-5] if stem.lower().endswith('_anno') else stem

def get_pairs(split):
    images = files_in(DATASET_ROOT / split / 'img')
    masks = files_in(DATASET_ROOT / split / 'mask')
    mask_map = {}
    for p in masks:
        key = canonical_mask_stem(p)
        if key in mask_map: raise ValueError(f'Duplicate mask key: {key}')
        mask_map[key] = p
    pairs, missing, used = [], [], set()
    for image in images:
        mask = mask_map.get(image.stem)
        if mask is None: missing.append(image.name)
        else: pairs.append((image, mask)); used.add(mask)
    extra = [p.name for p in masks if p not in used]
    if missing or extra: raise ValueError(f'{split}: missing masks={missing[:10]}, extra masks={extra[:10]}')
    if not pairs: raise ValueError(f'No paired samples in {split}')
    return pairs

test_pairs = get_pairs('test')
print(f'Verified test pairs: {len(test_pairs)}')


## 2. Load checkpoint

For custom layers, import/register the class first. For U-Net++, use the single-output inference/averaged-output model.


In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
model.summary()


## 3. Metrics

Macro averages are calculated across images. Boundary scores are based only on boundaries derived from binary ground truth and prediction.


In [ ]:
def inner_boundary(mask):
    mask = (np.asarray(mask) > 0).astype(np.uint8)
    eroded = cv2.erode(mask, np.ones((3,3), np.uint8), iterations=1, borderType=cv2.BORDER_CONSTANT, borderValue=0)
    return (mask-eroded).astype(np.uint8)

def ratio(a,b,empty=0.0): return float(a/b) if b else float(empty)

def scores(gt,pred,tolerance=2):
    gt=(np.asarray(gt).squeeze()>0).astype(np.uint8); pred=(np.asarray(pred).squeeze()>0).astype(np.uint8)
    if gt.shape != pred.shape: raise ValueError(f'Mask shape mismatch: {gt.shape} vs {pred.shape}')
    tp=int(np.logical_and(gt==1,pred==1).sum()); fp=int(np.logical_and(gt==0,pred==1).sum())
    fn=int(np.logical_and(gt==1,pred==0).sum()); tn=int(np.logical_and(gt==0,pred==0).sum())
    dice=ratio(2*tp,2*tp+fp+fn,1.0); iou=ratio(tp,tp+fp+fn,1.0)
    precision=ratio(tp,tp+fp,1.0 if tp+fn==0 else 0.0); recall=ratio(tp,tp+fn,1.0 if tp+fp==0 else 0.0)
    specificity=ratio(tn,tn+fp,1.0 if tn+fn==0 else 0.0); accuracy=ratio(tp+tn,tp+tn+fp+fn,1.0)
    gb=inner_boundary(gt); pb=inner_boundary(pred); k=2*max(0,int(tolerance))+1
    kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k)); gd=cv2.dilate(gb,kernel,iterations=1); pd=cv2.dilate(pb,kernel,iterations=1)
    ng=int(gb.sum()); npred=int(pb.sum())
    bp=ratio(int(np.logical_and(pb>0,gd>0).sum()),npred,1.0 if ng==0 else 0.0)
    br=ratio(int(np.logical_and(gb>0,pd>0).sum()),ng,1.0 if npred==0 else 0.0)
    bf1=ratio(2*bp*br,bp+br,0.0)
    return {'dice':dice,'iou':iou,'precision':precision,'recall':recall,'specificity':specificity,'pixel_accuracy':accuracy,'boundary_f1':bf1,'boundary_precision':bp,'boundary_recall':br}


## 4. Test inference and exports

Default preprocessing assumes RGB images divided by 255 and nearest-neighbour mask resizing. Copy the exact preprocessing from training before using results in a paper.


In [ ]:
rows=[]
for idx,(image_path,mask_path) in enumerate(test_pairs,1):
    bgr=cv2.imread(str(image_path),cv2.IMREAD_COLOR); mask_raw=cv2.imread(str(mask_path),cv2.IMREAD_GRAYSCALE)
    if bgr is None or mask_raw is None: raise IOError(f'Could not read pair: {image_path}, {mask_path}')
    image=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB); image=cv2.resize(image,(IMG_SIZE[1],IMG_SIZE[0]),interpolation=cv2.INTER_LINEAR)
    gt=cv2.resize(mask_raw,(IMG_SIZE[1],IMG_SIZE[0]),interpolation=cv2.INTER_NEAREST); image_float=image.astype(np.float32)/255.0
    output=model.predict(image_float[None,...],verbose=0)
    if isinstance(output,(list,tuple)): raise ValueError('Use model with single-output inference/averaged output')
    output=np.asarray(output)
    if output.ndim==4: prob=output[0,...,0]
    elif output.ndim==3: prob=output[0]
    else: raise ValueError(f'Unexpected output shape: {output.shape}')
    pred=(prob>=PRED_THRESHOLD).astype(np.uint8); gt_bin=(gt>0).astype(np.uint8)
    row={'image':image_path.name,**scores(gt_bin,pred,BOUNDARY_TOLERANCE_PX)}; rows.append(row)
    target=OUTPUT_DIR/'predicted_masks'/f'{image_path.stem}_prediction.png'
    if not cv2.imwrite(str(target),pred*255): raise IOError(f'Could not write {target}')
    if idx==1:
        fig,axes=plt.subplots(1,3,figsize=(15,5)); axes[0].imshow(image); axes[0].set_title('Input'); axes[1].imshow(gt_bin,cmap='gray'); axes[1].set_title('Ground truth'); axes[2].imshow(pred,cmap='gray'); axes[2].set_title('Prediction')
        for ax in axes: ax.axis('off')
        fig.tight_layout(); fig.savefig(OUTPUT_DIR/'first_test_visualization.png',dpi=200,bbox_inches='tight'); plt.show()
    print(f'[{idx:02d}/{len(test_pairs)}] {image_path.name}: Dice={row["dice"]:.4f}, IoU={row["iou"]:.4f}')
per_image=pd.DataFrame(rows); metric_cols=[c for c in per_image.columns if c!='image']
summary=pd.DataFrame({'metric':metric_cols,'mean':[per_image[c].mean() for c in metric_cols],'std':[per_image[c].std(ddof=1) if len(per_image)>1 else 0.0 for c in metric_cols],'median':[per_image[c].median() for c in metric_cols]})
per_image.to_csv(OUTPUT_DIR/'test_per_image_metrics.csv',index=False); summary.to_csv(OUTPUT_DIR/'test_summary_macro_mean_std.csv',index=False)
metadata={'dataset_root':str(DATASET_ROOT),'model_path':str(MODEL_PATH),'test_pairs':len(test_pairs),'image_size_hw':list(IMG_SIZE),'prediction_threshold':PRED_THRESHOLD,'boundary_tolerance_px':BOUNDARY_TOLERANCE_PX,'aggregation':'macro per-image mean and sample standard deviation','ground_truth_source':'test/mask; boundary maps derived from binary masks'}
(OUTPUT_DIR/'evaluation_metadata.json').write_text(json.dumps(metadata,indent=2))
print('Per-image CSV:',OUTPUT_DIR/'test_per_image_metrics.csv'); print('Aggregate CSV:',OUTPUT_DIR/'test_summary_macro_mean_std.csv'); display(summary)


## Publication check

Verify preprocessing matches training, checkpoint selection used validation only, outputs are foreground probabilities, threshold was frozen before testing, and all variants use the same split/evaluator. No new metric values are claimed until executed on actual checkpoints.
